In [ ]:
import pandas as pd

url = 'https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv';

dataset = pd.read_csv(url);
# Link para o código no Colab: https://colab.research.google.com/drive/1_XsssN5ddhY4Da_3tlOoMQeVIbA4kR7B?usp=sharing

In [ ]:
# 🔹 Mapeamento CORRETO (usando nomes completos das colunas)
COLUMN_MAP = {
    'Você ficou gripado no ano passado ?': 'ficou_gripado',
    'Você tomou vacina da gripe no ano passado?': 'vacina_gripe',
    '  Você frequentou no ano passado,  semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)  ': 'frequentou_ambientes_cheios',
    '  Você viajou no ano passado mais de 100 km de distância?  ': 'viajou_100km',
    '  Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?  ': 'alergia_vias_aereas',
    'Quantas horas você dormiu em média por noite no ano passado?': 'horas_sono',
    'Você praticou atividade física no ano passado?': 'atividade_fisica',
    'Você se alimentou de forma balanceada no ano passado?': 'alimentacao_balanceada',
    'Em média, quantas vezes você lavou as mãos por dia no ano passado?': 'lavou_maos_por_dia',
    'Na sua percepção, o seu nível de estresse no ano passado foi:': 'nivel_estresse',
}

# 🔹 Renomear dataset
ds_maped = dataset.rename(columns=COLUMN_MAP)

# 🔹 Definição de features
FEATURES = {
  "risk_exposure_score": ["frequentou_ambientes_cheios","viajou_100km",],
  "protection_score": ["vacina_gripe","lavou_maos_por_dia",],
  "health_score": ["atividade_fisica","alimentacao_balanceada","horas_sono",],
  "respiratory_vulnerability": ["alergia_vias_aereas","nivel_estresse",],
  "preventive_behavior_score": ["vacina_gripe","lavou_maos_por_dia","atividade_fisica","alimentacao_balanceada",],
  "lifestyle_balance": ["horas_sono","atividade_fisica","alimentacao_balanceada","nivel_estresse",],
  "immunity_behavior_index": ["horas_sono","alimentacao_balanceada","atividade_fisica","nivel_estresse",],
}

# 🔹 Target (string!)
TARGET = "ficou_gripado"

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 🔹 Encoder do target (fit uma única vez)
le = LabelEncoder()
le.fit(ds_maped[TARGET])

RANDOM_STATE = 42

K = 5

# 🔹 Loop nas FEATURES (nome + colunas)
for feature_name, feature_cols in FEATURES.items():

    # 🔹 Embaralha dataset
    df_shuffled = ds_maped.sample(
        frac=1,
        random_state=RANDOM_STATE,
    ).reset_index(drop=True)

    # 🔹 Split teste
    test_df = df_shuffled.iloc[:30].copy()
    y_test = le.transform(test_df[TARGET])  # usa transform (não fit)
    X_test = test_df[feature_cols]  # mantém como DataFrame (importante)

    # 🔹 Split treino
    train_df = df_shuffled.iloc[30:150].copy()
    y_train = le.transform(train_df[TARGET])
    X_train = train_df[feature_cols]

    # 🔹 Modelo KNN
    model = Pipeline(steps=[
        ("encoder", OrdinalEncoder()),  # converte categórico → numérico
        ("scaler", StandardScaler()),  # normaliza escala
        ("knn", KNeighborsClassifier(n_neighbors=K, weights="distance"))
    ])

    # 🔹 Treina
    model.fit(X_train, y_train)

    # 🔹 Predição
    y_pred = model.predict(X_test)

    # 🔹 Avaliação
    print(f"\nFeature group: {feature_name}")
    print("Colunas:", feature_cols)
    print("Acurácia:", accuracy_score(y_test, y_pred))
    print("\nRelatório:")
    print(classification_report(y_test, y_pred))
    print("\n========================\n")